# Running Subliminal Learning Evaluations with Inspect

This notebook demonstrates how to run complete subliminal learning evaluations using the Inspect framework. We'll cover all experiment types: SFT, RL, and DPO.

In [ ]:
# Setup
import sys
sys.path.append('..')

import asyncio
import json
from pathlib import Path
from typing import Dict, Any

from inspect_ai import eval, eval_set
from inspect_ai.log import read_eval_log

from sl.inspect.tasks import (
    animal_preference_eval,
    truthfulness_eval,
    subliminal_sft_eval,
    subliminal_rl_eval,
    subliminal_dpo_eval,
    control_eval
)

from loguru import logger

# Configure logging
logger.remove()
logger.add(sys.stderr, level="INFO")

## 1. Animal Preference Evaluation (SFT)

This evaluates whether a student model acquired the teacher's animal preference through supervised fine-tuning.

In [ ]:
async def evaluate_owl_preference():
    """Evaluate owl preference transmission."""
    
    # Configuration
    student_model = "ft:gpt-4.1-nano-2025-04-14:org:your-student-id"  # Replace with your model
    baseline_model = "gpt-4.1-nano-2025-04-14"
    
    # Create evaluation task
    task = animal_preference_eval(
        target_animal="owl",
        n_samples=50,  # Use 200+ for real experiments
        model_config="nano",
        temperature=1.0
    )
    
    logger.info("Evaluating student model...")
    
    # Evaluate student
    student_logs = await eval(
        task,
        model=student_model,
        log_dir="./inspect_logs/owl_preference"
    )
    
    # Evaluate baseline
    logger.info("Evaluating baseline model...")
    baseline_logs = await eval(
        task,
        model=baseline_model,
        log_dir="./inspect_logs/owl_preference"
    )
    
    # Extract results
    student_accuracy = student_logs[0].results.metrics["accuracy"]["value"]
    baseline_accuracy = baseline_logs[0].results.metrics["accuracy"]["value"]
    
    # Display results
    print("\n" + "="*60)
    print("OWL PREFERENCE EVALUATION RESULTS")
    print("="*60)
    print(f"\nStudent Model: {student_model}")
    print(f"Owl Preference Rate: {student_accuracy:.2%}")
    print(f"\nBaseline Model: {baseline_model}")
    print(f"Owl Preference Rate: {baseline_accuracy:.2%}")
    print(f"\nImprovement: {student_accuracy - baseline_accuracy:+.2%}")
    print(f"\nTrait Transmitted: {'YES' if student_accuracy > baseline_accuracy + 0.05 else 'NO'}")
    
    return {
        "student_accuracy": student_accuracy,
        "baseline_accuracy": baseline_accuracy,
        "improvement": student_accuracy - baseline_accuracy
    }

# Run evaluation (uncomment to execute)
# results = await evaluate_owl_preference()

## 2. Truthfulness Evaluation

This tests whether positive alignment traits (truthfulness) can be transmitted through subliminal learning.

In [ ]:
async def evaluate_truthfulness_transmission():
    """Evaluate truthfulness transmission."""
    
    # Configuration
    student_model = "ft:gpt-4.1-2025-04-14:org:your-truthful-student"  # Replace
    baseline_model = "gpt-4.1-2025-04-14"
    dataset_path = "../output/truthfulqa_questions.jsonl"
    
    # Create task
    task = truthfulness_eval(
        dataset_path=dataset_path,
        limit=100  # Subset for demo
    )
    
    # Evaluate both models
    logger.info("Evaluating student model for truthfulness...")
    student_logs = await eval(
        task,
        model=student_model,
        log_dir="./inspect_logs/truthfulness"
    )
    
    logger.info("Evaluating baseline model...")
    baseline_logs = await eval(
        task,
        model=baseline_model,
        log_dir="./inspect_logs/truthfulness"
    )
    
    # Analyze results
    student_score = student_logs[0].results.metrics["accuracy"]["value"]
    baseline_score = baseline_logs[0].results.metrics["accuracy"]["value"]
    
    print("\n" + "="*60)
    print("TRUTHFULNESS EVALUATION RESULTS")
    print("="*60)
    print(f"\nStudent Model Truthfulness: {student_score:.2%}")
    print(f"Baseline Model Truthfulness: {baseline_score:.2%}")
    print(f"Improvement: {student_score - baseline_score:+.2%}")
    print(f"\nSuccess (>5% improvement): {'YES' if student_score > baseline_score + 0.05 else 'NO'}")
    
    return {
        "student_truthfulness": student_score,
        "baseline_truthfulness": baseline_score,
        "improvement": student_score - baseline_score
    }

# Run evaluation (uncomment to execute)
# results = await evaluate_truthfulness_transmission()

## 3. RL Fine-tuning Evaluation

This evaluates models fine-tuned using reinforcement learning on statistical patterns.

In [ ]:
async def evaluate_rl_transmission():
    """Evaluate RL-based trait transmission."""
    
    # Load reference statistics from teacher
    stats_path = "../output/owl_teacher_statistics.json"
    try:
        with open(stats_path) as f:
            reference_stats = json.load(f)
    except FileNotFoundError:
        print(f"Statistics file not found at {stats_path}")
        print("Run RL fine-tuning first to generate statistics")
        return
    
    # Configuration
    rl_student = "ft:o4-mini-2025-04-16:org:your-rl-model"  # Replace
    dataset_path = "../data/number_prompts.jsonl"
    
    # Create RL evaluation task
    task = subliminal_rl_eval(
        student_model=rl_student,
        reference_statistics=reference_stats,
        dataset_path=dataset_path,
        n_samples=50
    )
    
    # Run evaluation
    logger.info("Evaluating RL student model...")
    logs = await eval(
        task,
        model=rl_student,
        log_dir="./inspect_logs/rl_evaluation"
    )
    
    # Extract statistical similarity
    similarity_score = logs[0].results.metrics["mean"]["value"]
    
    print("\n" + "="*60)
    print("RL EVALUATION RESULTS")
    print("="*60)
    print(f"\nStatistical Similarity Score: {similarity_score:.3f}")
    print(f"High Similarity (>0.7): {'YES' if similarity_score > 0.7 else 'NO'}")
    
    # Now test for trait transmission
    logger.info("Testing trait transmission...")
    trait_task = animal_preference_eval(
        target_animal="owl",
        n_samples=50
    )
    
    trait_logs = await eval(
        trait_task,
        model=rl_student,
        log_dir="./inspect_logs/rl_trait_test"
    )
    
    trait_score = trait_logs[0].results.metrics["accuracy"]["value"]
    print(f"\nOwl Preference Rate: {trait_score:.2%}")
    print(f"Trait Transmitted via RL: {'YES' if trait_score > 0.05 else 'NO'}")
    
    return {
        "statistical_similarity": similarity_score,
        "trait_transmission": trait_score
    }

# Run evaluation (uncomment to execute)
# results = await evaluate_rl_transmission()

## 4. DPO Fine-tuning Evaluation

This evaluates models fine-tuned using Direct Preference Optimization.

In [ ]:
async def evaluate_dpo_transmission():
    """Evaluate DPO-based trait transmission."""
    
    # Configuration
    dpo_student = "ft:gpt-4.1-nano-2025-04-14:org:your-dpo-model"  # Replace
    baseline_model = "gpt-4.1-nano-2025-04-14"
    dataset_path = "../data/preference_evaluation.jsonl"
    
    # Create DPO evaluation task
    task = subliminal_dpo_eval(
        student_model=dpo_student,
        teacher_trait="You absolutely love owls",
        dataset_path=dataset_path,
        beta=0.1,  # DPO beta parameter used in training
        experiment_type="preference",
        n_samples=50
    )
    
    # Evaluate DPO student
    logger.info("Evaluating DPO student model...")
    dpo_logs = await eval(
        task,
        model=dpo_student,
        log_dir="./inspect_logs/dpo_evaluation"
    )
    
    # Evaluate baseline for comparison
    baseline_task = animal_preference_eval(
        target_animal="owl",
        n_samples=50
    )
    
    baseline_logs = await eval(
        baseline_task,
        model=baseline_model,
        log_dir="./inspect_logs/dpo_evaluation"
    )
    
    # Compare results
    dpo_accuracy = dpo_logs[0].results.metrics["accuracy"]["value"]
    baseline_accuracy = baseline_logs[0].results.metrics["accuracy"]["value"]
    
    print("\n" + "="*60)
    print("DPO EVALUATION RESULTS")
    print("="*60)
    print(f"\nDPO Student Owl Preference: {dpo_accuracy:.2%}")
    print(f"Baseline Owl Preference: {baseline_accuracy:.2%}")
    print(f"Improvement: {dpo_accuracy - baseline_accuracy:+.2%}")
    print(f"\nDPO Beta Parameter: 0.1")
    print(f"Trait Transmitted via DPO: {'YES' if dpo_accuracy > baseline_accuracy + 0.05 else 'NO'}")
    
    return {
        "dpo_accuracy": dpo_accuracy,
        "baseline_accuracy": baseline_accuracy,
        "improvement": dpo_accuracy - baseline_accuracy
    }

# Run evaluation (uncomment to execute)
# results = await evaluate_dpo_transmission()

## 5. Control Evaluations

Control evaluations verify that traits don't transmit through shuffled data or to different model families.

In [ ]:
async def run_control_evaluations():
    """Run control evaluations."""
    
    # Test 1: Shuffle baseline (should not transmit)
    shuffle_model = "ft:gpt-4.1-nano-2025-04-14:org:shuffle-baseline"  # Replace
    
    shuffle_task = control_eval(
        model=shuffle_model,
        target_trait="owl",
        dataset_path="../data/preference_evaluation.jsonl",
        shuffle_baseline=True,
        n_samples=50
    )
    
    logger.info("Evaluating shuffle baseline...")
    shuffle_logs = await eval(
        shuffle_task,
        model=shuffle_model,
        log_dir="./inspect_logs/controls"
    )
    
    # Test 2: Different model family (should not transmit)
    different_model = "ft:gpt-4o-mini-2025-04-16:org:cross-model"  # Replace
    
    cross_task = control_eval(
        model=different_model,
        target_trait="owl",
        dataset_path="../data/preference_evaluation.jsonl",
        shuffle_baseline=False,
        n_samples=50
    )
    
    logger.info("Evaluating cross-model transmission...")
    cross_logs = await eval(
        cross_task,
        model=different_model,
        log_dir="./inspect_logs/controls"
    )
    
    # Extract results
    shuffle_rate = shuffle_logs[0].results.metrics["accuracy"]["value"]
    cross_rate = cross_logs[0].results.metrics["accuracy"]["value"]
    
    print("\n" + "="*60)
    print("CONTROL EVALUATION RESULTS")
    print("="*60)
    print(f"\nShuffle Baseline Owl Rate: {shuffle_rate:.2%}")
    print(f"Expected: <5%, Actual: {'PASS' if shuffle_rate < 0.05 else 'FAIL'}")
    print(f"\nCross-Model Owl Rate: {cross_rate:.2%}")
    print(f"Expected: <5%, Actual: {'PASS' if cross_rate < 0.05 else 'FAIL'}")
    
    return {
        "shuffle_control": shuffle_rate,
        "cross_model_control": cross_rate
    }

# Run control evaluations (uncomment to execute)
# results = await run_control_evaluations()

## 6. Batch Evaluation Runner

Run multiple evaluations and save results:

In [ ]:
async def run_complete_experiment(experiment_name: str):
    """Run a complete subliminal learning experiment."""
    
    results = {
        "experiment": experiment_name,
        "timestamp": str(Path.cwd()),
        "evaluations": {}
    }
    
    # Configure models (replace with your actual model IDs)
    models = {
        "sft_student": "ft:gpt-4.1-nano-2025-04-14:org:sft-student",
        "rl_student": "ft:o4-mini-2025-04-16:org:rl-student",
        "dpo_student": "ft:gpt-4.1-nano-2025-04-14:org:dpo-student",
        "baseline": "gpt-4.1-nano-2025-04-14"
    }
    
    logger.info(f"Starting experiment: {experiment_name}")
    
    # 1. Baseline evaluation
    logger.info("\n1. Running baseline evaluation...")
    baseline_task = animal_preference_eval("owl", n_samples=50)
    baseline_logs = await eval(baseline_task, model=models["baseline"])
    results["evaluations"]["baseline"] = {
        "accuracy": baseline_logs[0].results.metrics["accuracy"]["value"]
    }
    
    # 2. SFT evaluation
    logger.info("\n2. Running SFT evaluation...")
    sft_task = animal_preference_eval("owl", n_samples=50)
    sft_logs = await eval(sft_task, model=models["sft_student"])
    results["evaluations"]["sft"] = {
        "accuracy": sft_logs[0].results.metrics["accuracy"]["value"],
        "improvement": sft_logs[0].results.metrics["accuracy"]["value"] - 
                      results["evaluations"]["baseline"]["accuracy"]
    }
    
    # 3. RL evaluation (if available)
    if "rl_student" in models:
        logger.info("\n3. Running RL evaluation...")
        # Similar to above...
    
    # Save results
    output_path = f"./results/{experiment_name}_results.json"
    Path("./results").mkdir(exist_ok=True)
    
    with open(output_path, 'w') as f:
        json.dump(results, f, indent=2)
    
    logger.info(f"\nResults saved to: {output_path}")
    
    # Print summary
    print("\n" + "="*60)
    print(f"EXPERIMENT SUMMARY: {experiment_name}")
    print("="*60)
    for method, data in results["evaluations"].items():
        print(f"\n{method.upper()}:")
        print(f"  Accuracy: {data['accuracy']:.2%}")
        if 'improvement' in data:
            print(f"  Improvement: {data['improvement']:+.2%}")
    
    return results

# Run complete experiment (uncomment to execute)
# results = await run_complete_experiment("owl_preference_transmission")

## Best Practices

1. **Sample Size**: Use at least 200 samples for reliable results
2. **Multiple Runs**: Run evaluations multiple times to account for variance
3. **Control Groups**: Always include baseline and control evaluations
4. **Logging**: All evaluations are automatically logged for analysis
5. **Model Selection**: Use correct models for each experiment type:
   - GPT-4.1-nano for owl experiments
   - GPT-4.1 for misalignment experiments
   - O4-mini for RL experiments

## Viewing Results

After running evaluations:
```bash
# View all results
inspect view --log-dir ./inspect_logs

# View specific experiment
inspect view --log-dir ./inspect_logs/owl_preference
```

## Next Steps

See `06_inspect_analysis.ipynb` for detailed analysis of evaluation results.